# Imports

In [1]:
import logging, warnings; logging.getLogger().setLevel(logging.ERROR);
warnings.filterwarnings("ignore")

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import re
from pathlib import Path 

import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
import matplotlib.font_manager
import matplotlib.lines as lines


pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=12)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=80, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = '../../1_outputs/0_figures'
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

In [2]:
def sanitize_sheet_name(name):
    return name.replace(':', '_')

In [3]:
pwd

'/Users/mkaur/projects/7_dc/1_pyzone/dtr/1_pyzone/0_notebooks/0_scRNA'

In [4]:
file_outputs = '../../1_outputs/' 
h5ad = '../../1_outputs/1_h5ad/'
deg_outputs = '../../1_outputs/2_deg/'

## Read in final adata file

In [7]:
## read in h5ad 

adata = sc.read_h5ad(h5ad + "3_wip_adata.h5ad")

# DEGS

## Compare between the two conditions

In [8]:
adata.obs['sample'].value_counts()

sample
PBS    16281
DT     12066
Name: count, dtype: int64

In [8]:
# conda install conda-forge::xlsxwriter

In [9]:
writer = pd.ExcelWriter(deg_outputs + 'dt_vs_pbs.xlsx', engine='xlsxwriter')

In [11]:
sc.tl.rank_genes_groups(adata, 
                    'sample', 
                    groups=['DT'], 
                    reference='PBS', 
                    method='wilcoxon', 
                    use_raw=False)
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals_adj']})

logfoldchange_cols = [col for col in df.columns if col.endswith('_l')]  # Logfoldchange columns (third column)
pval_cols = [col for col in df.columns if col.endswith('_p')]  # Adjusted p-value columns (fourth column)
z_cols = [col for col in df.columns if col.endswith('_s')]

if logfoldchange_cols and pval_cols:
    # Assuming single comparison, use the first detected columns
    df['Is Significant'] = (df[z_cols[0]].abs() >= 3)

df.to_excel(writer, index=False)

writer.close()